In [20]:
from langchain_groq import ChatGroq
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain.agents import create_agent
from dotenv import load_dotenv
from sqlalchemy import create_engine
from pydantic import BaseModel, Field
import os

load_dotenv()

True

In [21]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [22]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DATABASE = os.getenv("POSTGRES_DATABASE")

connection_string = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
)

engine = create_engine(connection_string)

db = SQLDatabase(engine=engine)

print(db.dialect)
print(db.get_usable_table_names())

db.run("SELECT * FROM project;")


postgresql
['manager', 'project']


"[(3, 'Procurement Management', None, None, None, None), (4, None, None, None, None, None), (5, 'HIMS', None, None, None, None), (2, 'Payroll Management', 'Test Description', None, None, 'ON_HOLD'), (1, 'HR Management', 'Hr management Demo', datetime.date(2026, 1, 31), datetime.date(2026, 1, 3), 'ON_HOLD')]"

In [23]:
toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()
agent = create_agent(
    model = model,
    tools=tools
)

In [33]:
# question = "Tell me the names of all the managers in manager table"
# question = "Remove the manager whose name is bikash and is not assigned to any project"
# question = "Create a manager named bikash in manager table and assign him to procurement project"
question = "Show me all the projects along with the managers assigned to them"


result =agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

result["messages"][-1].content


'Here are all the projects in the database together with any manager that is assigned to each project.  \nIf a project has no manager yet, the manager fields are shown as **NULL**.\n\n| project_id | project_name          | project_description | start_date | end_date   | status   | manager_id | manager_name |\n|------------|----------------------|---------------------|------------|------------|----------|------------|--------------|\n| 1          | HR Management        | Hr management Demo  | 2026‑01‑03 | 2026‑01‑31 | ON_HOLD  | NULL       | NULL |\n| 2          | Payroll Management   | Test Description    | NULL       | NULL       | ON_HOLD  | 7          | Bikash |\n| 3          | Procurement Management| NULL               | NULL       | NULL       | NULL     | 8          | bikash |\n| 4          | NULL                 | NULL                | NULL       | NULL       | NULL     | NULL       | NULL |\n| 5          | HIMS                 | NULL                | NULL       | NULL       | N

In [35]:
sql_prompt = """
Write a PostgreSQL SQL query for:
How many rows are there in the manager table?
"""

sql = model.invoke(sql_prompt).content
sql


'Here’s a simple PostgreSQL query that returns the total number of rows in the **manager** table:\n\n```sql\nSELECT COUNT(*) AS total_rows\nFROM   manager;\n```\n\n- `COUNT(*)` counts every row, regardless of whether any column contains `NULL`.\n- The result will be a single row with the column name `total_rows` (you can rename it to anything you like).'